<a href="https://colab.research.google.com/github/Gabriel-Arsego/Group-Project-for-ADMPA/blob/main/Preprocessing%20and%20rough%20variable%20selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train Dataset

So, for this part what I did was basically load and clean this huge dataset in smaller pieces, since my previous tries were crashing in colab. It was using too much RAM and wasn't able to perform the cleaning well. I uploaded the files directly to colab and it read them from there.

My main goal here was to reduce memory use, by reading the data in smaller chuncks. After doing that I combined them all together again.

The chuncks' size was 20,000. This helped prevent RAM crashes (which happened a lot for me, before finally trying this)

Modifications made:
- the column with "id" was removed, because it's just an identifier (for the test dataset later, I saved the ids so we can use them in the final submission)
- I kept only the numerical variables to make modeling easier
- filled missing values with the median off the columns
- "downcast=" converted big numbers to smaller data types

In [1]:
import pandas as pd

chunk_size = 20000
chunks = []

for chunk in pd.read_csv("train_v3.csv", chunksize=chunk_size, low_memory=False):

    # Drop ID
    if "id" in chunk.columns:
        chunk = chunk.drop(columns=["id"])

    # Keep only numeric columns
    chunk = chunk.select_dtypes(include=['int64', 'float64'])

    # Fill missing values
    chunk = chunk.fillna(chunk.median())

    # Downcast to reduce memory
    for col in chunk.columns:
        if chunk[col].dtype == 'int64':
            chunk[col] = pd.to_numeric(chunk[col], downcast='integer')
        elif chunk[col].dtype == 'float64':
            chunk[col] = pd.to_numeric(chunk[col], downcast='float')

    chunks.append(chunk)

# Combine all cleaned chunks
df = pd.concat(chunks, ignore_index=True)

print(df.shape)

(80000, 742)


Here I selected the 50 variables that are most related to predicting loss. For that, correlation was used. This reduced the size of the dataset considerably while keeping the most important predictors.

In [2]:
corr = df.corr()["loss"].abs().sort_values(ascending=False)

top_features = corr.head(50).index

df = df[top_features]

print(df.shape)

(80000, 50)


Here I just create the cleaned train file:

In [3]:
df.to_csv("cleaned_train_final.csv", index=False)

This next line counts the total number of missing values in the dataset, so 0 is the expected response:

In [4]:
df.isna().sum().sum()

np.int64(0)

# Test dataset

Here I created a list of the selected features from the training dataset, so that both datasets have the same variables for modeling.

In [5]:
features = list(top_features)
features.remove("loss")  # test doesn't have loss

Loading the test dataset:

In [8]:
test = pd.read_csv("test__no_lossv3.csv", low_memory=False)

The next 3 lines are basically saving the ids from the test set in a separate file (which I said I was going to do earlier - so that we can use it to create the final submission) and removing them from it as well so that we can do modeling.

In [9]:
test_ids = test["id"]

In [17]:
test_ids.to_csv("test_ids.csv", index=False)

In [10]:
test = test.drop(columns=["id"])

From this part on, I'm just making sure that the test dataset is clean and matches the training dataset:

In [11]:
test = test.select_dtypes(include=['int64', 'float64'])

In [12]:
test = test.fillna(test.median())

In [13]:
for col in test.columns:
    if test[col].dtype == 'int64':
        test[col] = pd.to_numeric(test[col], downcast='integer')
    elif test[col].dtype == 'float64':
        test[col] = pd.to_numeric(test[col], downcast='float')

In [14]:
for col in features:
    if col not in test.columns:
        test[col] = 0

test = test[features]

In [15]:
test.to_csv("cleaned_test_final.csv", index=False)

I did some extra tests, because I was worried of having less columns or that the number of rows was different from the original test dataset.

Testing to see if it works = checking column names and column order

In [16]:
train = pd.read_csv("cleaned_train_final.csv")
test = pd.read_csv("cleaned_test_final.csv")

train_features = list(train.columns)
train_features.remove("loss")

test_features = list(test.columns)

print("Match:", train_features == test_features)

Match: True


In [18]:
print(test.shape)

(25471, 49)


In [19]:
import pandas as pd

test = pd.read_csv("test__no_lossv3.csv", low_memory=False)

print(test.shape)

(25471, 762)


In [20]:
test = pd.read_csv("test__no_lossv3.csv", low_memory=False)
print(test.shape)

test_ids = test["id"]
print(len(test_ids))

(25471, 762)
25471
